# 06.9 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/09-market-basket/exercises.ipynb)

Exercise 1 is on paper first.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

baskets = [
    ["tea", "cake", "mug"],
    ["tea", "cake"],
    ["tea", "mug"],
    ["cake", "candle"],
    ["tea", "cake", "card"],
    ["candle", "card"],
    ["tea", "cake", "candle"],
    ["mug", "card"],
]
items = sorted({i for b in baskets for i in b})
onehot = pd.DataFrame([[i in b for i in items] for b in baskets], columns=items)
onehot.index = [f"B{k}" for k in range(1, 9)]
onehot.astype(int)

## Exercise 1: by hand

From the eight baskets compute support, confidence and lift for **cake → tea** and for **card → mug**. Is lift symmetric (is lift(A → B) equal to lift(B → A))? Is confidence?

In [ ]:
# your code here

## Exercise 2: Germany

Repeat the lesson's analysis for German invoices (min_support 0.05, no postage). Show the top 8 rules by lift. Are they the same *kind* of rules as in France?

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"], **{"Customer ID": sales["Customer ID"].astype(int)})
print(f"{len(sales):,} clean transaction lines, {sales['Customer ID'].nunique():,} customers")

from mlxtend.frequent_patterns import apriori, association_rules

def basket_matrix(df, drop_postage=True):
    """One row per invoice, one True/False column per product."""
    b = df.groupby(["Invoice", "Description"])["Quantity"].sum().unstack(fill_value=0) > 0
    if drop_postage:
        b = b.drop(columns=[c for c in b.columns if "POSTAGE" in str(c)])
    return b

def show(rules, n=8):
    out = rules.sort_values("lift", ascending=False).head(n).copy()
    out["if bought"] = out["antecedents"].apply(lambda s: " + ".join(sorted(s)))
    out["then also"] = out["consequents"].apply(lambda s: " + ".join(sorted(s)))
    return out[["if bought", "then also", "support", "confidence", "lift"]].round(3)

# your code here

## Exercise 3: do the French rules hold over time?

Split the French invoices into the first half of 2011 (before 1 July) and the second half. Mine rules on each half (min_support 0.05, lift > 1). How many of the first half's rules also appear in the second half?

In [ ]:
# your code here